# Chapter 1: Basic Prompt Structure

**Technique:** Prompt anatomy with system, user, and assistant roles

**Contents**
* [Lesson: The Messages API](#lesson)
* [Exercises](#exercises)
* [Example Playground](#example-playground)

In [ ]:
import { getCompletion, MODEL, client } from "../lib/helpers.js";

## Lesson: The Messages API

Every Claude API call requires three top level fields:

* `model` (string): pinned model ID (e.g. `claude-sonnet-4-6`)
* `max_tokens` (number): hard upper bound on output tokens
* `messages` (array): ordered list of conversation turns

Each element of `messages` is `{ role, content }` where `role` is either `"user"` or `"assistant"`. Two rules the API enforces:

1. **Roles must alternate.** You cannot have two consecutive `user` or `assistant` turns.
2. **The first message must be `"user"`.**

The `system` prompt is a separate top level string, not a message turn. Use it to set Claude's persona, constraints, or output format. The helper `getCompletion(prompt, system?)` wires all of this up; `system` defaults to `""`.

This is the same shape you'll wire into a backend service: construct a `messages` array from your application's conversation state, pass it to the API, stream or await the reply.

In [ ]:
console.log(await getCompletion("What is the time complexity of binary search?"));

In [ ]:
console.log(await getCompletion("What year was Node.js first released?"));

### Malformed requests (intentional errors)

The two cells below violate the API contract and will throw when executed. Run them to see the error messages; they illustrate exactly what the API enforces.

In [ ]:
const res = await client.messages.create({
  model: MODEL, max_tokens: 2000, temperature: 0,
  messages: [{ content: "Hi Claude, how are you?" }], // missing role, throws an error
});
console.log(res.content[0].text);

In [ ]:
const res = await client.messages.create({
  model: MODEL, max_tokens: 2000, temperature: 0,
  messages: [
    { role: "user", content: "What is REST?" },
    { role: "user", content: "Also, what is gRPC?" },
  ],
});
console.log(res.content[0].text);

### System prompts

The `system` prompt is delivered to Claude before any user turn. Use it to supply context and rules that should govern the entire conversation: persona, output format, domain constraints, safety guardrails. The user turn then carries the actual request.

The example below configures Claude as a requirements reviewer that only asks clarifying questions, never proposes solutions. This pattern is useful for guardrailing Claude to a specific workflow role inside a larger application.

In [ ]:
const SYSTEM_PROMPT = "You are a senior engineer doing requirements review. Respond ONLY with clarifying questions that surface missing requirements. Do not propose a solution.";
console.log(await getCompletion("Build me a rate limiter.", SYSTEM_PROMPT));

## Exercises

### Exercise 1.1: Count to three

**Task**: Edit `PROMPT` so that Claude's response contains the numbers 1, 2, and 3.

**Success criteria**: the response text includes all of `"1"`, `"2"`, and `"3"`.

In [ ]:
import { includesAll, grade } from "../lib/grading.js";

const PROMPT = "Count from 1 to 3, listing each number.";

const response = await getCompletion(PROMPT);
const gradeExercise = (text) => includesAll(text, ["1", "2", "3"]);
grade(response, gradeExercise(response));

In [ ]:
import { exercise_1_1_hint } from "../hints.js";
console.log(exercise_1_1_hint);

### Exercise 1.2: Terse TODO answers

**Scenario**: you're building a CLI tool that surfaces quick engineering guidance. You want answers that are terse and action oriented, formatted as a `// TODO` style code comment, not prose.

**Task**: Edit `SYSTEM_PROMPT` so that Claude answers as a terse senior engineer who replies only with a `// TODO` style code comment.

**Success criteria**: the response contains `"TODO"`.

In [ ]:
import { includesAny, grade } from "../lib/grading.js";

const SYSTEM_PROMPT = "You are a terse senior engineer. Reply ONLY with a single-line // TODO: code comment giving the action to take. No prose, no explanation.";
const PROMPT = "How should I handle a failed database connection on startup?";

const response = await getCompletion(PROMPT, SYSTEM_PROMPT);
const gradeExercise = (text) => includesAny(text, ["TODO"]);
grade(response, gradeExercise(response));

In [ ]:
import { exercise_1_2_hint } from "../hints.js";
console.log(exercise_1_2_hint);

## Common mistakes

* **First turn is not `"user"`**: the API rejects any `messages` array whose first element has `role: "assistant"`. Always start with a user turn.
* **Putting constraints in the user turn instead of `system`**: rules buried in the user message can be overridden or forgotten partway through the conversation. Stable constraints belong in the system prompt where they persist across all turns.

**Reflect**: when would you put a constraint in the system prompt vs the user message? Consider what changes from conversation to conversation vs what is fixed for all users of your application.

## Example Playground

Use the cell below to experiment freely. Swap in any prompt. The graders are not active here.

In [ ]:
console.log(await getCompletion("Explain the CAP theorem in two sentences."));